# CONDA toxicity detection — context window k=3

This notebook fine-tunes **DeBERTa-v3-base** on the CONDA dataset with **k=3 previous messages** as conversational context.

**Output:** a saved model directory on Google Drive that can be loaded by the demo website.

**Runtime:** ~75 min on a Colab T4 GPU. Set `Runtime > Change runtime type > GPU` before running.

## 1. Setup — install deps, mount Drive

In [1]:
# Install dependencies (Colab usually has torch + transformers; sentencepiece is needed for DeBERTa-v3)
!pip install -q sentencepiece protobuf

In [2]:
from pathlib import Path

K = 3

# Local Colab disk (NOT Drive). Will be lost when the session ends — make sure
# to download the .tar.gz at the end of the notebook.
SAVE_ROOT = Path('/content/conda_ksweep')
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = SAVE_ROOT / f'k{K}_model'
CKPT_DIR  = SAVE_ROOT / f'k{K}_checkpoints'
MODEL_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)
print(f'Will save final model to: {MODEL_DIR}')
print(f'Training checkpoints in : {CKPT_DIR}')

Will save final model to: /content/conda_ksweep/k3_model
Training checkpoints in : /content/conda_ksweep/k3_checkpoints


In [3]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, accuracy_score, classification_report
import json, time, gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
assert device.type == 'cuda', 'GPU not available! Runtime > Change runtime type > GPU (T4)'

Device: cuda


## 2. Load CONDA dataset

Upload the three CONDA CSV files: `CONDA_train.csv`, `CONDA_valid.csv`, `CONDA_test.csv` (Ctrl+click to select all three at once).

In [4]:
from google.colab import files
uploaded = files.upload()

train_df = pd.read_csv('CONDA_train.csv')
valid_df = pd.read_csv('CONDA_valid.csv')
print(f'Train: {train_df.shape}, Valid: {valid_df.shape}')
train_df.head()

Saving CONDA_test.csv to CONDA_test.csv
Saving CONDA_train.csv to CONDA_train.csv
Saving CONDA_valid.csv to CONDA_valid.csv
Train: (26921, 10), Valid: (8974, 10)


,Id,matchId,conversationId,utterance,chatTime,playerSlot,playerId,intentClass,slotClasses,slotTokens
0,11263,697,3193,wow!,76,0,ANTS IN MY EYES JOHNSON,O,O,"wow (O),"
1,13741,843,3809,WTF,1563,5,M.k,O,T,"WTF (T),"
2,22125,1412,6199,wpe wpe,2853,1,Acqua Ragia,O,O O,"wpe (O), wpe (O),"
3,6453,439,1875,hahaha,1038,0,juicebox,O,O,"hahaha (O),"
4,9644,601,2713,wtf,1661,5,KAIST.Shadows,O,T,"wtf (T),"


## 3. Preprocessing

- Map intent labels `E/I/A/O` → integers `0/1/2/3`
- Sort by `conversationId` and `chatTime` so prior messages are well-defined
- Load DeBERTa-v3-base tokenizer

In [5]:
LABEL2ID = {'E': 0, 'I': 1, 'A': 2, 'O': 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

train_df['label'] = train_df['intentClass'].map(LABEL2ID)
valid_df['label'] = valid_df['intentClass'].map(LABEL2ID)

# Sort so context lookup is correct
for df in (train_df, valid_df):
    df.sort_values(['conversationId', 'chatTime'], inplace=True, kind='mergesort')
    df.reset_index(drop=True, inplace=True)

print('Label distribution (train):')
print(train_df['label'].value_counts().sort_index().rename(index=ID2LABEL))

Label distribution (train):
label
E     3528
I     1692
A     1719
O    19982
Name: count, dtype: int64


In [6]:
MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Sanity check: speaker tags like 'P3:' should tokenize cleanly
test = 'P3: ez mid [SEP] P7: report him'
print(f'Tokenization of {test!r}:')
print(' ', tokenizer.tokenize(test))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenization of 'P3: ez mid [SEP] P7: report him':
  ['▁P', '3', ':', '▁ez', '▁mid', '[SEP]', '▁P', '7', ':', '▁report', '▁him']


## 4. Contextualized dataset (k=3)

For each target message, prepend up to **3 preceding messages** from the same `conversationId`, separated by `[SEP]`. Each message is prefixed with a speaker tag `P{playerSlot}:` so the model can distinguish speakers.

In [7]:
MAX_K = 10  # we pre-compute up to 10 even if K is smaller

def build_context_index(df, max_k=10):
    ctx = [[] for _ in range(len(df))]
    for _, group in df.groupby('conversationId', sort=False):
        idxs = group.index.tolist()
        for pos, i in enumerate(idxs):
            ctx[i] = idxs[max(0, pos - max_k):pos]
    return ctx

train_context_idx = build_context_index(train_df, max_k=MAX_K)
valid_context_idx = build_context_index(valid_df, max_k=MAX_K)
n_with_ctx = sum(1 for c in train_context_idx if c)
print(f'Train rows with at least 1 prior msg: {n_with_ctx}/{len(train_df)} ({100*n_with_ctx/len(train_df):.1f}%)')

Train rows with at least 1 prior msg: 17512/26921 (65.0%)


In [8]:
SEP = ' [SEP] '

class CONDAContextDataset(Dataset):
    """
    Builds: 'P3: prev1 [SEP] P7: prev2 [SEP] ... [SEP] P3: target'
    k=0 => target alone (no speaker tag) — matches the original baseline.
    """
    def __init__(self, df, context_idx, tokenizer, k, max_length=256):
        self.df = df
        self.context_idx = context_idx
        self.tokenizer = tokenizer
        self.k = k
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def _fmt(self, row, include_speaker):
        text = str(row['utterance'])
        return f"P{int(row['playerSlot'])}: {text}" if include_speaker else text

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.k == 0:
            input_str = self._fmt(row, include_speaker=False)
        else:
            ctx_rows = self.context_idx[idx][-self.k:]
            if ctx_rows:
                parts = [self._fmt(self.df.iloc[j], True) for j in ctx_rows]
                parts.append(self._fmt(row, True))
                input_str = SEP.join(parts)
            else:
                input_str = self._fmt(row, True)
        enc = self.tokenizer(
            input_str, padding='max_length', truncation=True,
            max_length=self.max_length, return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(int(row['label']), dtype=torch.long),
        }

MAX_LENGTH = 128 if K == 0 else 256
train_dataset = CONDAContextDataset(train_df, train_context_idx, tokenizer, K, MAX_LENGTH)
valid_dataset = CONDAContextDataset(valid_df, valid_context_idx, tokenizer, K, MAX_LENGTH)

# Show what a sample input looks like
demo_i = next((i for i, c in enumerate(train_context_idx) if len(c) >= min(K,3)), 0)
sample = train_dataset[demo_i]
decoded = tokenizer.decode(sample['input_ids'], skip_special_tokens=False)
print(f'Sample input (k={K}, label={ID2LABEL[sample["labels"].item()]}):')
print(decoded[:400])

Sample input (k=3, label=O):
P1: wtf [SEPA] TA? [SEPA] u srsly?[SEP] P6: sad spec [SEPA] noes cape for him[SEP] P7: wat [SEPA] that one i cant even run[SEP] P1: why alyway hit me [SEPA] what did i do[PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD]


## 5. Load DeBERTa-v3-base + define metrics

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    torch_dtype=torch.float32,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded ({n_params:,} parameters)')

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

Model loaded (184,425,220 parameters)


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_per = f1_score(labels, preds, average=None, zero_division=0, labels=[0,1,2,3])
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_E': f1_per[0],
        'f1_I': f1_per[1],
        'f1_A': f1_per[2],
        'f1_O': f1_per[3],
    }

## 6. Train

Checkpoints save to Google Drive. If Colab disconnects, rerun this cell and it picks up where it left off.

In [11]:
training_args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Resume from existing checkpoint if any
ckpts = [p for p in CKPT_DIR.glob('checkpoint-*') if p.is_dir()]
resume = bool(ckpts)
if resume:
    print(f'Resuming from existing checkpoint(s): {[p.name for p in ckpts]}')

t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=resume)
elapsed_min = (time.time() - t0) / 60
print(f'\nTraining done in {elapsed_min:.1f} min')
print(f'Final train loss: {train_result.training_loss:.4f}')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 E,F1 I,F1 A,F1 O
1,0.497309,0.482519,0.862826,0.662327,0.793720,0.413712,0.511927,0.929951
2,0.321700,0.420037,0.900713,0.778325,0.826221,0.669145,0.670103,0.947832
3,0.268487,0.380162,0.913862,0.824977,0.838998,0.735178,0.774711,0.951020
4,0.189099,0.394072,0.911856,0.825866,0.838923,0.733927,0.780870,0.949746
5,0.154788,0.415427,0.912748,0.826696,0.844762,0.730067,0.781770,0.950183


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye


Training done in 36.7 min
Final train loss: 0.3538


## 7. Final evaluation on the validation set

In [12]:
eval_metrics = trainer.evaluate()
print('Validation metrics (best model):')
for kk, v in eval_metrics.items():
    if isinstance(v, float):
        print(f'  {kk:30s} {v:.4f}')

Validation metrics (best model):
  eval_loss                      0.4154
  eval_accuracy                  0.9127
  eval_f1_macro                  0.8267
  eval_f1_E                      0.8448
  eval_f1_I                      0.7301
  eval_f1_A                      0.7818
  eval_f1_O                      0.9502
  eval_runtime                   43.4397
  eval_samples_per_second        206.5850
  eval_steps_per_second          6.4690
  epoch                          5.0000


In [13]:
# Detailed per-class report
preds_output = trainer.predict(valid_dataset)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids
print(classification_report(
    y_true, y_pred,
    target_names=['E', 'I', 'A', 'O'],
    digits=4, zero_division=0,
))

              precision    recall  f1-score   support

           E     0.8409    0.8487    0.8448      1183
           I     0.8279    0.6529    0.7301       582
           A     0.7950    0.7690    0.7818       580
           O     0.9410    0.9596    0.9502      6629

    accuracy                         0.9127      8974
   macro avg     0.8512    0.8075    0.8267      8974
weighted avg     0.9110    0.9127    0.9111      8974



## 8. Save model + metadata to Drive

Final model goes to `{MODEL_DIR}`. The demo website loads it via:
```python
AutoModelForSequenceClassification.from_pretrained('/path/to/k3_model')
```

In [14]:
# Save best model + tokenizer in HuggingFace format
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

# Save metrics + config that the demo site can read
meta = {
    'k': K,
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'train_minutes': elapsed_min,
    'final_train_loss': float(train_result.training_loss),
    'eval_metrics': {kk: float(v) for kk, v in eval_metrics.items() if isinstance(v, (int, float))},
    'label2id': LABEL2ID,
    'id2label': ID2LABEL,
    'sep_token': SEP.strip(),
    'speaker_tag_format': 'P{playerSlot}:',
}
with open(MODEL_DIR / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Saved model to {MODEL_DIR}')
print('Contents:')
for p in sorted(MODEL_DIR.iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f'  {p.name:30s} {size_mb:8.2f} MB')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to /content/conda_ksweep/k3_model
Contents:
  config.json                        0.00 MB
  meta.json                          0.00 MB
  model.safetensors                737.73 MB
  tokenizer.json                     8.34 MB
  tokenizer_config.json              0.00 MB
  training_args.bin                  0.01 MB


In [15]:
# Optional: delete intermediate checkpoints to free Drive space (comment out if you want to keep them)
import shutil
for ckpt in CKPT_DIR.glob('checkpoint-*'):
    if ckpt.is_dir():
        shutil.rmtree(ckpt)
        print(f'Removed {ckpt}')

Removed /content/conda_ksweep/k3_checkpoints/checkpoint-6732
Removed /content/conda_ksweep/k3_checkpoints/checkpoint-8415


## Done

The model for **k=3** is saved at `{MODEL_DIR}`.

Next: run the notebook for the next k value, then the demo website can compare predictions across all trained models.

In [16]:
# Save the trained model to Colab local disk (bypasses Drive entirely), then download
import tarfile, json, os
from google.colab import files

LOCAL_MODEL_DIR = f'/content/k{K}_model'
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

# Save to /content (Colab's local disk, ~100GB free, NOT Drive)
trainer.save_model(LOCAL_MODEL_DIR)
tokenizer.save_pretrained(LOCAL_MODEL_DIR)

# Save metadata
meta = {
    'k': K,
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'label2id': LABEL2ID,
    'id2label': ID2LABEL,
    'sep_token': SEP.strip(),
    'speaker_tag_format': 'P{playerSlot}:',
}
# Include eval metrics if they're in scope
try:
    meta['eval_metrics'] = {kk: float(v) for kk, v in eval_metrics.items() if isinstance(v, (int, float))}
except NameError:
    pass

with open(f'{LOCAL_MODEL_DIR}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved to', LOCAL_MODEL_DIR)
print('Contents:')
for p in sorted(os.listdir(LOCAL_MODEL_DIR)):
    size_mb = os.path.getsize(f'{LOCAL_MODEL_DIR}/{p}') / 1e6
    print(f'  {p:30s} {size_mb:8.2f} MB')

# Archive and download
archive_path = f'/content/k{K}_model.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(LOCAL_MODEL_DIR, arcname=f'k{K}_model')

print(f'\nArchive: {archive_path} ({os.path.getsize(archive_path)/1e6:.1f} MB)')
print('Starting download...')
files.download(archive_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/k3_model
Contents:
  config.json                        0.00 MB
  meta.json                          0.00 MB
  model.safetensors                737.73 MB
  tokenizer.json                     8.34 MB
  tokenizer_config.json              0.00 MB
  training_args.bin                  0.01 MB

Archive: /content/k3_model.tar.gz (586.7 MB)
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('/content/k3_model.tar.gz', '/content/drive/MyDrive/k3_model.tar.gz')
print('Copied to Drive')

Mounted at /content/drive
Copied to Drive
